# M60 · Putting it together: a full run on a new sequence, and the plan for your project

**Part 4 · week 8 · about 10 h core**

## 한국어 요약

이 모듈에서는 지금까지 배운 과정 전체(자료 확인, 등록, 발췌, 위치 결정, 상대 재배치, 단층면해)를 새 자료인 2018년 3월 8일(UTC) 곡성 지진 일곱 개에 처음부터 끝까지 적용하고, 연구 과제인 순천 지진의 계획을 세웁니다. 먼저 무거운 계산을 시작하기 전에 입력 파일, 지진별 관측소 수, 시간대(한국 시간과 UTC), 초동 극성이 있는 발췌, 메모리와 코어 사용량을 점검하는 사전 점검 함수(preflight)를 정답을 아는 합성 자료로 작성합니다. 이 함수로 곡성 자료와 PocketQuake 발췌를 점검한 뒤 PocketQuake로 전체 과정을 실행하고, 지진마다 위치, 깊이, 단층면해와 그 불확실성을 담은 짧은 결과표를 만듭니다. 가장 깊은 지진에 대해 위상별 교차 거리를 계산해 HYPOINVERSE가 각 관측소에서 어떤 파를 예측했는지도 확인합니다. 이어서 아무것도 바꾸지 않은 재실행과 설정 하나(HYPOINVERSE 위상 가중 방식)를 바꾼 실행을 비교합니다. 마지막으로 순천 지역의 자료 목록(연도별 지진 수와 관측소 수, 카탈로그 깊이 분포, 내려받기 계획)과 Kwak et al. (2022)의 주장을 검증 가능한 문장으로 정리한 표를 만들고, 연구 계획서(ANALYSIS_PLAN.md)와 판단 기준(criteria.yaml)을 자리만 채운 글이 통과하지 못하는 점검 함수로 확인한 뒤 지도교수의 승인 서명을 받아 고정합니다.

## What you will be able to do

- Write a preflight check that catches missing inputs, too few stations, a time-zone slip, picks without usable polarities and a core or memory request above the server's budget, and test it on made-up folders whose problems are known
- Run the whole chain (stage, pick, locate, relocate, mechanisms) on a new dataset through PocketQuake, each heavy step after a dry run and a passed preflight
- Build one results table with the uncertainty of every location and the quality of every mechanism, and say what each uncertainty means
- Rerun a stage unchanged, then with one setting changed, and measure what the change moved
- Make the data inventory of the Suncheon project from the catalog, the station lists and the download plan, turn the claims of the published Suncheon study into testable statements, and freeze an analysis plan with its criteria after the mentor's sign-off, with an audit that placeholder text cannot pass

## Before you start

- Modules: M01, M03, M21, M30, M31, M32, M41, M51
- Reading: Kwak, Kim, Sheen & Kim (2022), the whole paper (for the claims table of task 6); Nosek, Ebersole, DeHaven & Mellor (2018), the sections on what a preregistration is and why it helps (for the analysis plan)
- Needs: synthetic, catalog, stp, pocketquake
- Key terms (see the glossary): preflight check (사전 점검), dry run (드라이 런, 시험 실행), KST (한국 표준시), UTC (협정 세계시), time zone (시간대), first-motion polarity (초동 극성), phase pick (위상 피킹, 도달 시각 읽기), CPU core (CPU 코어), memory (RAM; 메모리), thread (스레드), PocketQuake (지진 분석 도구 모음), cluster (PocketQuake; 클러스터 (PocketQuake)), HYPOINVERSE (진원 결정 프로그램), HypoDD (이중차분 진원 재결정 프로그램), SKHASH (단층면해 결정 프로그램), location errors (ERH and ERZ; 위치 오차 (ERH와 ERZ)), mechanism quality (단층면해 품질 등급), bootstrap (부트스트랩), checksum (SHA-256 fingerprint; 체크섬), run log (실행 기록), JSON Lines (JSON 줄 형식), deterministic computation (결정론적 계산), phase weight (위상 가중치), results table (결과 표), station coverage (관측소 분포), data inventory (자료 목록), station epoch (관측소 운영 기간, 에포크), claims table (주장 정리표), testable claim (검증 가능한 주장), hypothesis (가설), analysis plan (분석 계획서), prior exposure (사전 노출), pass criteria (판단 기준), plan amendment (계획 수정 기록), lab book (연구 노트), MD5 checksum (MD5 체크섬), preregistration (사전 등록), cuspid (cuspid (SKHASH 지진 번호)), epicenter (진앙), head wave (선두파), first arrival (최초 도달파 (초동)), crossover distance (교차 거리), mentor sign-off (지도교수 승인 서명)

## Passing this module

Each task is done when its `st.check(...)` cell prints PASS. The pass criteria are frozen in [criteria/M60.yaml](../../../.sgtl/criteria/M60.yaml).

## AI use

Follow [docs/ai_policy.md](../../../docs/ai_policy.md) and log every AI conversation in `work/ai_log/M60.md`.

---

© 2026 Minseong Seo (SGTL, Seoul National University) and contributors · code MIT · text CC BY 4.0 · cite CITATION.cff

In [ ]:
# Parameters: the only cell with values you may change. Run it first, then the setup cell.
DATASET = "gokseong2018"                  # your STP download of the seven Gokseong earthquakes (sgtl data fetch)
PQ_SLUG = "gokseong_m60"                  # the name of your new PocketQuake cluster (sgtl never reuses a name)
PQ_CORES = 3                              # cores of each PocketQuake run (at most 4 on the shared server)
PQ_MEM_GB = 8.0                           # memory cap of the runs in GB (HypoDD needs at least 6)
MECH_MEM_GB = 12.0                        # memory cap of the mechanism stage (SKHASH with S/P ratios needs about 9)
MIN_STATIONS = 8                          # Part 1: fewest stations with a vertical record that an event needs
MIN_POLARITIES = 8                        # Part 1: fewest usable polarities (PocketQuake's fm_npolmin)
MIN_WEIGHT = 0.3                          # Part 1: smallest polarity size used (fm_min_polarity_weight)
MIN_PROB = 0.5                            # Part 1: smallest pick probability of a polarity (fm_min_pick_prob)
BOOT_N = 200                              # Part 3: replicas of the relocation bootstrap
BOOT_SEED = 0                             # Part 3: seed of the bootstrap
CHANGED_SCHEME = "distance"               # Part 4: the one setting you change (PocketQuake's phs_weight_scheme)
PROJECT_SEQUENCE = "suncheon"             # Part 5: the named catalog subset of your project
PROJECT_DATASET = "suncheon_area"         # Part 5: its STP dataset (about 5 GB: not fetched in this module)
RADII_KM = (30.0, 50.0, 100.0)            # Part 5: station-coverage radii
YEARS = tuple(range(2002, 2027))          # Part 5: years of the inventory
FIGURE_DIR = "figures"                    # your figure files go here, next to this notebook
RESULT_DIR = "results"                    # your result files go here, next to this notebook

In [ ]:
import sgtl_tutorial as st
S = st.init("M60")

import contextlib
import datetime as dt
import hashlib
import io
import json
import re
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yaml

Path(FIGURE_DIR).mkdir(exist_ok=True)
Path(RESULT_DIR).mkdir(exist_ok=True)
FOLDER = st.data.root() / DATASET                                   # your download of the Gokseong records
RUN_DIR = st.pq.pipeline_dir() / "pipeline" / "runs" / PQ_SLUG      # PocketQuake's results for your cluster
RUNS_LOG = st.data.repo_path("work", ".sgtl", "runs.jsonl")         # sgtl's log of your PocketQuake runs
PROJECT = st.data.repo_path("project")                              # your project folder (sgtl new project)
print("dataset folder:", FOLDER)
print("PocketQuake results of your cluster:", RUN_DIR)


def sha256_of(path):
    """SHA-256 of a file's bytes."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def show_lines(path, words):
    """Print the lines of a file that contain any of the words (with their line numbers)."""
    for number, line in enumerate(Path(path).read_text(encoding="utf-8").splitlines(), 1):
        if any(w in line for w in words):
            print(f"{Path(path).name}:{number}: {line.strip()}")

## How this notebook works

The same five steps as in the earlier modules, for every task: read the **Contract**, write a **prediction**,
replace `raise NotImplementedError(...)` under `# YOUR CODE HERE` with your code, run the check cell, and answer
the **Explain** prompt in 3-5 sentences of your own. `st.hint("M60-T1", 1)` shows a first hint; hints 2 and 3
unlock after failed checks. The checks try your functions on made-up data whose answers are known from how they
were built (`st.reroll("M60-T1")` draws new ones). Tasks 2-6 also read your own run products, your project files
and the run log; the checks never print the values they compare with.

This module has two halves. **Parts 1-4** run the whole chain of modules M03-M51 on a dataset you have not seen,
the seven Gokseong earthquakes of 8 March 2018 (UTC), in the order of every module: a function first, tested on
made-up data with a known answer (task 1); then the same function on the real data and the production tool next
to it (task 2); then one table of results with their uncertainties (task 3); then the production tool again, an
unchanged rerun and one changed setting (task 4). **Parts 5-6** start your research project on the Suncheon
earthquakes: a data inventory (task 5) and a plan with testable claims and frozen criteria (task 6).
The research portfolio also requires a separate independent Sangju track and validated template matching in
both tracks. The [project overview](../../../docs/project.md) gives the phased schedule; the
[Sangju guide](../../../docs/sangju_project.md) explains its independent selection and plan. Task 6 audits the
Suncheon files; the Sangju plan and evidence receive a separate mentor review.

This module needs your STP download `gokseong2018` (`sgtl data fetch gokseong2018`, a dry run first, then
`--apply`; about 340 MB), the course tools (`bin/sgtl-install tools`), the station metadata
(`bin/sgtl-install metadata`) and, for Part 6, your project folder (`sgtl new project`). The PocketQuake runs of
Part 2 take about 15 minutes on 3 cores; never use more than 4 cores on the shared server, and run one heavy job
at a time.

You may ask an AI tool to explain an error message or a concept. It may not write the task functions, your
predictions, your Explain answers, your claims table or your analysis plan
([docs/ai_policy.md](../../../docs/ai_policy.md)); log AI conversations in `work/ai_log/M60.md`.

## Part 0 · A new dataset, and two of your own functions (about 20 min)

The seven earthquakes of your download, from the KMA catalog that ships with the course (UTC). They are not in a
named sequence of the course: the dataset was chosen as a small, new test of the whole chain.

In [ ]:
kma = st.data.kma_catalog()
SAC_ROOT = FOLDER / "stp_download" / "SAC"
folders = sorted(p.name for p in SAC_ROOT.iterdir() if p.is_dir()) if SAC_ROOT.is_dir() else []
seven = kma[kma["origin_utc"].dt.strftime("%Y%m%d%H%M%S").isin(folders)].reset_index(drop=True)
print(f"{len(folders)} earthquake folders in your download {DATASET!r}; in the KMA catalog:")
print(seven[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].to_string(index=False))

Two functions of module M01 come back here: `haversine_km` (task M01-T1, the great-circle distance in km on a
sphere of radius `EARTH_RADIUS_KM` = 6371 km) and `kst_to_utc_id` (task M01-T2, the UTC time ID `YYYYMMDDHHMMSS`
of a Korean clock reading). Paste both into the next cell as they are in your M01 notebook, with the line
`EARTH_RADIUS_KM = 6371.0` above them (they need numpy and `datetime as dt`, both imported above), and run it.
Tasks 1 and 5 may call them.

In [ ]:
# Paste EARTH_RADIUS_KM = 6371.0, your haversine_km and your kst_to_utc_id of module M01 below this line.

In [ ]:
missing = [name for name in ("EARTH_RADIUS_KM", "haversine_km", "kst_to_utc_id") if name not in globals()]
if missing:
    raise NameError(f"{' and '.join(missing)} not found: paste your functions of module M01 into the cell above and "
                    "run it (finish task M01-T1 and task M01-T2 of module M01 first).")
print("haversine_km, from 35.0 N 127.0 E to 35.1 N 127.0 E:", round(float(haversine_km(35.0, 127.0, 35.1, 127.0)), 3),
      "km;  kst_to_utc_id(2018, 1, 1, 3, 0, 0):", kst_to_utc_id(2018, 1, 1, 3, 0, 0))

Two functions of module M30 come back in Part 3: `direct_time` (task M30-T1, the travel time of the direct wave in a
flat layered model) and `head_wave` (task M30-T2, the head wave along one boundary, with its critical and crossover
distances). Paste both into the next cell as they are in your M30 notebook (`head_wave` calls `direct_time`) and
run it.

In [ ]:
# Paste your direct_time and your head_wave of module M30 below this line.

In [ ]:
missing = [name for name in ("direct_time", "head_wave") if name not in globals()]
if missing:
    raise NameError(f"{' and '.join(missing)} not found: paste your functions of module M30 into the cell above and "
                    "run it (finish task M30-T1 and task M30-T2 of module M30 first).")
KIM1983 = st.data.velocity_model("kim1983")
print("kim1983:", KIM1983.to_dict("records"))
print("your head_wave along the 15-km boundary, P, source at 5 km: critical and crossover distance (km):",
      [round(float(v), 1) for v in head_wave(100.0, 5.0, KIM1983, "P", 1)[1:]])

## Part 1 · Look before you run (about 30 min, then task 1)

A **preflight** check (glossary `preflight`) is a short program that looks at the inputs and the requested
resources of a heavy run before the run starts, and stops it when something is wrong. A run that fails after
twenty minutes on four cores wastes the shared server; a run that finishes on the wrong inputs is worse, because
its products look fine. Every problem below happened to someone in this lab:

- **Inputs present.** `sgtl data fetch` writes the catalog of a dataset to `event_catalog/event_catalog.csv`, in
  Korean time (KST), and the records of each earthquake to `stp_download/SAC/<UTC time ID>/<band>/`, one SAC file
  per station and component named `<start>.<NET>.<STA>.<CHAN>.sac`.
- **Time zone.** KST is UTC + 9 h (glossary `kst`, `utc`, `time_zone`; module M01). If a catalog time in KST is
  read as UTC, the records start 9 hours too late and hold no earthquake (module M03, Part 7). Such a mistake shows
  in the folder names: a folder named with the Korean clock time instead of the UTC one.
- **Station counts per event.** A location needs stations around the source; a station counts here when it has a
  record of the vertical component (channel code ending in `Z`) in any band. One station recorded by a broadband
  seismometer (HH) and an accelerometer (HG) is still one station.
- **Picks with polarity.** A focal mechanism needs **phase picks** (glossary `phase_pick`) of P with a first-motion
  **polarity** (glossary `polarity`; module M21). PocketQuake's pick files hold the polarity of PhaseNet+ as a
  signed number between -1 (down) and 1 (up); its mechanism stage keeps a P polarity only when its size is at least
  `fm_min_polarity_weight` and its pick probability at least `fm_min_pick_prob`, and SKHASH tries an event only when
  it has at least `fm_npolmin` of them. The lines of PocketQuake's settings file, as they are in your clone:

In [ ]:
show_lines(st.pq.pipeline_dir() / "pipeline" / "config.py",
           ["fm_min_polarity_weight:", "fm_min_pick_prob:", "fm_npolmin:", "phs_weight_scheme:", "num_cores:"])
print(f"\nthis notebook: MIN_WEIGHT={MIN_WEIGHT}, MIN_PROB={MIN_PROB}, MIN_POLARITIES={MIN_POLARITIES}")

- **Core and memory budget.** On the shared server a heavy run uses at most 4 **CPU cores** (glossary `cpu_core`)
  and, in this course, less than 16 GB of **memory** (glossary `memory`). `sgtl pq` refuses to start unless the
  server has at least 1.5 times the requested memory cap free (`MemAvailable` in `/proc/meminfo`, in kB).

### M60-T1 · A preflight check · core · about 90 min

**Goal:** write `preflight(folder, picks=None, *, cores=1, mem_gb=8.0, mem_available_gb=None, min_stations=8,
min_polarities=8, min_weight=0.3, min_prob=0.5)`: seven checks of a dataset folder (and of a pick table, when one
is given) and of a resource request, and the counts behind them.

**Given:** Part 1; your `kst_to_utc_id` (task M01-T2); pandas and `pathlib`. Not allowed: the course package and
PocketQuake.

**Contract:**
- Inputs: `folder`, a dataset folder as above (a `Path` or a string; read file *names* only, never change or add a
  file); `picks`, None or a DataFrame like PocketQuake's pick files (columns `Event_ID`, `Network`, `Station`,
  `Phase`, `Time`, `Probability`, `Polarity`, `Amplitude`; `Event_ID` is the UTC time ID of the event folder, maybe
  read as an integer; `Polarity` may be missing or NaN); do not change it. `cores`, `mem_gb`: the request.
  `mem_available_gb`: the free memory in GB (2**30 bytes), or None to read `MemAvailable` from `/proc/meminfo`.
- The event folders are the folders of `stp_download/SAC/` whose names are 14 digits. The catalog's `Second` is a
  whole number.
- Output: a tuple `(report, counts)`.
  `report`: a DataFrame with the columns `check`, `status` (`"PASS"`, `"FAIL"` or `"SKIP"`) and `detail` (a short
  text of your choice), one row per check in this order:
  1. `catalog`: PASS when `event_catalog/event_catalog.csv` exists and has the columns Year, Month, Day, Hour,
     Minute and Second.
  2. `time_zone`: FAIL when the catalog is missing, or when an event folder is named with a catalog time written
     in KST as it stands; otherwise PASS.
  3. `events`: PASS when the event folder names are exactly the catalog's times converted to UTC (no folder
     missing, none extra); FAIL otherwise (also without a catalog).
  4. `stations`: PASS when there is at least one event folder and every one has at least `min_stations` stations
     (distinct network and station codes with a vertical-component file in any band).
  5. `polarity`: SKIP when `picks` is None; FAIL when it has no `Polarity` column; otherwise PASS when every event
     folder has at least `min_polarities` usable polarities: P picks of that event with a polarity whose absolute
     value is at least `min_weight` and a probability at least `min_prob`.
  6. `cores`: PASS when 1 <= `cores` <= 4.
  7. `memory`: PASS when `mem_gb` <= 16 and the free memory is at least 1.5 x `mem_gb`.
  `counts`: a DataFrame indexed by the event folder names (sorted), with the columns `stations` (the station count
  of check 4) and `polarities` (the usable polarities of check 5; NaN for every event when the polarity check is
  not PASS or FAIL by count, that is when `picks` is None or has no `Polarity` column).

**Done when** `st.check("M60-T1", func=preflight)` passes: 20 made-up folders, each built with a known problem
or none (Korean clock times around midnight and New Year included).

**Stuck?** `st.hint("M60-T1", 1)`

In [ ]:
# Before you code: which of the seven checks do you expect to catch a problem in your real Gokseong folder, and which
# in PocketQuake's picks of the seven earthquakes, once they exist? For the smallest earthquakes (ML 1.1-1.3), how
# many usable polarities do you expect compared with the largest (ML 2.0)?
st.predict("M60-T1", """

""")

In [ ]:
def preflight(folder, picks=None, *, cores=1, mem_gb=8.0, mem_available_gb=None, min_stations=8,
              min_polarities=8, min_weight=0.3, min_prob=0.5):
    """Seven checks of a dataset folder, a pick table and a resource request, before a heavy run.

    folder : dataset folder (event_catalog/event_catalog.csv in KST, stp_download/SAC/<UTC id>/<band>/*.sac)
    picks  : None or a DataFrame like PocketQuake's pick files (not changed)
    cores, mem_gb : the request; mem_available_gb : free memory in GB, None = read /proc/meminfo
    returns : (report, counts) as in the Contract
    """
    # Step 1: the catalog, and its times converted to UTC time IDs (your kst_to_utc_id); also the times as written.
    # Step 2: the event folders; compare their names with both sets of IDs.
    # Step 3: per folder, the distinct stations with a vertical-component file in any band.
    # Step 4: per folder, the usable polarities (when picks has a Polarity column).
    # Step 5: the request against the budget; build the two tables.
    # YOUR CODE HERE
    raise NotImplementedError("M60-T1: write your code here")

In [ ]:
st.check("M60-T1", func=preflight)

**Explain (3-5 sentences):** Which made-up problem was hardest to catch, and why? Why does the time-zone check
compare folder names with the catalog times as written, and not only with the converted ones? Why is a station
with an HH and an HG record counted once? Why does a polarity of -0.8 count as usable, and one of 0.2 not?

## Part 2 · The whole chain on the Gokseong earthquakes (about 60 min, half of it while the runs go; task 2)

A full run is the chain of the earlier modules, in this order: the dataset (module M03), registering it as a
**PocketQuake cluster** (glossary `pocketquake_cluster`; module M21), the stations within 100 km and their records,
picking with PhaseNet+ (M21), absolute locations with **HYPOINVERSE** (glossary `hypoinverse`; M32), catalog
differential times and **HypoDD** (glossary `hypodd`; M41), the cross-correlation times and the relocation with
them (M41), and the mechanisms with **SKHASH** (glossary `skhash`; M51). Every heavy step gets a **dry run**
(glossary `dry_run`) first and a passed preflight. PocketQuake's station list ends at 100 km, and HYPOINVERSE
locates with the P and S picks of those stations. Which wave its layered model predicts at each station, the direct
wave or a **head wave** (glossary `head_wave`) along a boundary below the source, depends on the depth; Part 3
computes it for your deepest event with your functions of module M30. No location uses the Moho head waves Pn or Sn.

First the dataset itself: what the fetch would still do (a dry run), and the file check of module M03.

In [ ]:
st.datafetch.fetch(DATASET)               # a dry run: nothing is downloaded

In [ ]:
st.datafetch.verify(DATASET)

Your preflight of the folder, with the request of the PocketQuake runs. The next cell stops the notebook when a
check that matters for the runs fails (the polarity check is SKIP: there are no picks yet).

In [ ]:
data_check = preflight(FOLDER, cores=PQ_CORES, mem_gb=PQ_MEM_GB, min_stations=MIN_STATIONS,
                       min_polarities=MIN_POLARITIES, min_weight=MIN_WEIGHT, min_prob=MIN_PROB)
print(data_check[0].to_string(index=False))
print("\nstations with a vertical record, per event folder:")
print(data_check[1]["stations"].to_string())
blocking = data_check[0][(data_check[0]["status"] == "FAIL") & (data_check[0]["check"] != "polarity")]
if len(blocking):
    raise RuntimeError("The preflight failed: " + "; ".join(blocking["check"] + ": " + blocking["detail"])
                       + ". Fix the input (or wait for free memory) before any heavy run.")

### M60-T2 · The preflight on your data, and the whole chain · core · about 120 min

**Goal:** write `read_picks(pick_dir)`: all of PocketQuake's pick files of a run as one table, with the combined
fingerprint that sgtl logs for them. Then run the chain below, check the picks with your preflight before the
mechanism stage, and report everything in a dictionary `chain_report`.

**Given:** the run cells below (read each one before you run it); Part 2; module M41 (Part 7) for the run log
`work/.sgtl/runs.jsonl` and its SHA-256 fingerprints (glossary `checksum`, `run_log`, `json_lines`). Not allowed
in `read_picks`: the course package and PocketQuake.

**Contract (the function):**
- Input: `pick_dir`, a folder (a `Path` or a string) with one `<event>_picks.csv` per event (the columns of
  Part 1; a file may have a header and no rows) and maybe other files. Do not change it.
- Output: a tuple `(picks, fingerprint)`. `picks`: one DataFrame of every row of the `*.csv` files of the folder,
  files in the order of their names and rows in file order, with the columns `Event_ID` (as text, so that it
  equals the event folder name), `Network`, `Station`, `Phase`, `Time`, `Probability`, `Polarity`, `Amplitude`,
  and a fresh index. `fingerprint`: the SHA-256 (64 hexadecimal characters) of the text made of one line
  `<sha256 of the file's bytes>  <file name>` (two spaces, a newline at the end) per `.csv` file, in the order of
  their names: the rule by which sgtl logs pick files in the run log.

**Contract (the report):** `chain_report = {"preflight": preflight, "read_picks": read_picks, "folder": str(FOLDER),
"slug": PQ_SLUG, "run_dir": str(RUN_DIR), "settings": {"cores": PQ_CORES, "mem_gb": PQ_MEM_GB, "mech_mem_gb":
MECH_MEM_GB, "min_stations": MIN_STATIONS, "min_polarities": MIN_POLARITIES, "min_weight": MIN_WEIGHT, "min_prob":
MIN_PROB}, "data_check": data_check, "picks_sha256": PICKS_SHA, "picks_check": picks_check}`.

**Done when** `st.check("M60-T2", answer=chain_report)` passes: your `read_picks` on made-up pick folders and your
preflight once more on made-up folders; then the check runs your preflight again on your folder and on your pick
files, recomputes the fingerprint and finds it in the run log, and finds a successful run of every stage from
`stations` to `focal_mechanism`.

**Stuck?** `st.hint("M60-T2", 1)`

In [ ]:
# Before you code: your preflight counts the usable polarities of every event with PocketQuake's three settings
# (Part 1). SKHASH reports for every mechanism how many P polarities it used (num_p_pol in mechanisms.csv). Will the
# two counts agree for all seven earthquakes? If not, which will be larger, and which step of PocketQuake would make
# the difference?
st.predict("M60-T2", """

""")

In [ ]:
def read_picks(pick_dir):
    """All pick files of a PocketQuake run as one table, and their combined fingerprint.

    pick_dir : folder with <event>_picks.csv files (not changed)
    returns  : (picks DataFrame with Event_ID as text, 64-character SHA-256 fingerprint)
    """
    # Step 1: the .csv files of the folder, sorted by name; read each with Event_ID as text.
    # Step 2: one table with a fresh index.
    # Step 3: the fingerprint of the files, by the rule of the Contract.
    # YOUR CODE HERE
    raise NotImplementedError("M60-T2: write your code here")

**Register the dataset** as your cluster (a dry run first; module M21 did the same for Sangju). If the cluster
exists already (you ran this cell before), the next cell leaves it alone.

In [ ]:
st.pq.stage(DATASET, PQ_SLUG)                                                     # dry run

In [ ]:
if not (st.pq.pipeline_dir() / "pipeline" / "clusters" / f"{PQ_SLUG}.py").is_file():
    assert st.pq.stage(DATASET, PQ_SLUG, apply=True) == 0

**Stations, records, picks and HYPOINVERSE** (about 2 minutes on 3 cores; a dry run first):

In [ ]:
st.pq.run(PQ_SLUG, "stations", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB)    # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "stations", "hypoinverse", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

**The picks, before the mechanism stage.** Your `read_picks` reads them; your preflight checks them with the
request of the mechanism stage (one core, `MECH_MEM_GB`). A FAIL in the polarity row does not stop the chain:
SKHASH skips an event with too few polarities, and the row tells you which ones before you wait for it.

In [ ]:
picks, PICKS_SHA = read_picks(RUN_DIR / "picks")
picks_check = preflight(FOLDER, picks, cores=1, mem_gb=MECH_MEM_GB, min_stations=MIN_STATIONS,
                        min_polarities=MIN_POLARITIES, min_weight=MIN_WEIGHT, min_prob=MIN_PROB)
print(picks_check[0][["check", "status"]].to_string(index=False))

**The relocation stages** (module M41): `ph2dt` and `dtct` take a second; `rereference` and `xcorr` take about
10 minutes on 3 cores (start them, then read Part 3); `dtcc` a few seconds.

In [ ]:
assert st.pq.run(PQ_SLUG, "ph2dt", "dtct", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

In [ ]:
assert st.pq.run(PQ_SLUG, "rereference", "xcorr", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

In [ ]:
assert st.pq.run(PQ_SLUG, "dtcc", "dtcc", cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0

**The mechanisms** (module M51; one core, about a minute):

In [ ]:
st.pq.run(PQ_SLUG, "focal_mechanism", "focal_mechanism", cores=1, mem_gb=MECH_MEM_GB)          # dry run

In [ ]:
assert st.pq.run(PQ_SLUG, "focal_mechanism", "focal_mechanism", cores=1, mem_gb=MECH_MEM_GB, apply=True) == 0

The report (nothing is printed until the check has run):

In [ ]:
chain_report = {"preflight": preflight, "read_picks": read_picks, "folder": str(FOLDER), "slug": PQ_SLUG,
                "run_dir": str(RUN_DIR),
                "settings": {"cores": PQ_CORES, "mem_gb": PQ_MEM_GB, "mech_mem_gb": MECH_MEM_GB,
                             "min_stations": MIN_STATIONS, "min_polarities": MIN_POLARITIES,
                             "min_weight": MIN_WEIGHT, "min_prob": MIN_PROB},
                "data_check": data_check, "picks_sha256": PICKS_SHA, "picks_check": picks_check}

In [ ]:
st.check("M60-T2", answer=chain_report)

Now compare. Your usable polarities next to SKHASH's count for each event, and your station counts next to the
stations PocketQuake kept (its station list of the cluster) and the readings HYPOINVERSE weighted (the `num`
column of its summary file):

In [ ]:
FM_DIR = RUN_DIR / "3.FocalMech" / "kim1983"
mech = pd.read_csv(FM_DIR / "mechanisms.csv", dtype={"event_id": str})
used = pd.read_csv(RUN_DIR / "station_table" / "used_stations_100km.csv")
hypo = st.pq.read_sum(PQ_SLUG, "kim1983")
compare = picks_check[1].rename(columns={"stations": "your stations", "polarities": "your usable polarities"})
compare["SKHASH num_p_pol"] = compare.index.map(mech.drop_duplicates("event_id").set_index("event_id")["num_p_pol"])
compare["SKHASH solutions"] = compare.index.map(mech["event_id"].value_counts()).fillna(0).astype(int)
if len(hypo) == len(compare):       # HYPOINVERSE ids follow the catalog order, which is the order of the folders
    compare["HYPOINVERSE readings"] = hypo["num"].to_numpy()
print(compare.to_string())
print(f"\nPocketQuake's station list of the cluster: {len(used)} stations (one sensor per station, within 100 km)")
show_lines(st.pq.pipeline_dir() / "pipeline" / "config.py", ["sensor_precedence:", "radius_km:"])

**Explain (3-5 sentences):** Use the printed table. Did your usable polarities and SKHASH's num_p_pol agree for
every event, and what does that say about the rules you wrote? Which events have the fewest polarities, and what
did SKHASH give for them (read the solutions column)? How does PocketQuake's station list compare with your station
counts, and which settings (the printed lines) decide what it keeps? What does one HYPOINVERSE reading count, and
why can there be more readings than stations for one event and fewer for another?

## Part 3 · One table of results, with their uncertainties (about 30 min, then task 3)

A result without its uncertainty cannot be compared with anything. The chain gives three kinds, and they do not
mean the same thing:

- HYPOINVERSE's `ERH` and `ERZ` (km) are the horizontal and vertical **location errors** (glossary
  `location_errors`; module M32) from the covariance matrix of one linearized solution: about one standard error,
  for the velocity model taken as exact.
- The **bootstrap** (glossary `bootstrap`; module M41) reruns HypoDD on resampled differential times; PocketQuake
  writes, per event, the half-width of the middle 95 % of the replicas in east, north and depth (`ex95`, `ey95`,
  `ez95`, in meters) to `bootstrap_errors.csv`. It measures how the relative positions scatter with the data, not
  how far the whole cluster may be from the truth.
- SKHASH grades each mechanism A to D (**mechanism quality**, glossary `mechanism_quality`; module M51) and gives
  the uncertainty of its fault plane in degrees (`fault_plane_uncertainty`); an event may get several solutions.

A **results table** (glossary `results_table`) puts one row per event with every number and its uncertainty, so
that a reader can see at once what is known and how well. First the bootstrap of your relocation (a few seconds on
3 cores), then copies of the four files the table is made from, with their fingerprints:

In [ ]:
assert st.pq.bootstrap(PQ_SLUG, "dtcc", n=BOOT_N, seed=BOOT_SEED, cores=PQ_CORES, mem_gb=PQ_MEM_GB, apply=True) == 0
REGION = sorted((RUN_DIR / "1.HypoInv" / "kim1983").glob("*.sum"))[0].stem      # PocketQuake's name of the cluster
PRODUCTS = Path(RESULT_DIR) / "M60_products"
PRODUCTS.mkdir(exist_ok=True)
sources = {"hypoinverse.sum": RUN_DIR / "1.HypoInv" / "kim1983" / f"{REGION}.sum",
           "hypoDD.reloc": RUN_DIR / "2.HypoDD" / "02.dt.cc" / "hypoDD.reloc",
           "bootstrap_errors.csv": RUN_DIR / "2.HypoDD" / "02.dt.cc" / "bootstrap_errors.csv",
           "mechanisms.csv": FM_DIR / "mechanisms.csv"}
for name, source in sources.items():
    shutil.copy2(source, PRODUCTS / name)
    print(f"{name:22s} SHA-256 {sha256_of(PRODUCTS / name)}  (copy of {source.relative_to(RUN_DIR)})")
print("\nthe first lines of each copy:")
for name in sources:
    print(f"--- {name}")
    print("\n".join((PRODUCTS / name).read_text(encoding="utf-8").splitlines()[:2]))

### M60-T3 · One results table · core · about 90 min

**Goal:** write `results_table(sum_file, reloc_file, boot_file, mech_file)`: one row per located earthquake, with
the HYPOINVERSE location's errors, the relocation with its bootstrap errors, and the mechanism with its quality.

**Given:** the four copies above and their first lines; modules M32 (the summary file), M41 (the relocation file:
24 columns separated by spaces, `ID LAT LON DEPTH X Y Z EX EY EZ YR MO DY HR MI SC MAG NCCP NCCS NCTP NCTS RCC RCT
CID`) and M51 (`mechanisms.csv`). Not allowed: the course package (`st.pq.read_sum` and friends read these files
for you) and PocketQuake.

**Contract:**
- Inputs: four file paths (do not change the files). `sum_file`: HYPOINVERSE's summary file in PocketQuake's
  comma layout (a header line, then one line per event; column names padded with spaces; `ID-NUM` is the event's
  number, which PocketQuake calls cuspid). `reloc_file`: HypoDD's relocations (no header; `ID` is the cuspid; some
  located events may be missing). `boot_file`: PocketQuake's bootstrap errors (a first line starting with `#`, then
  a header; `id` is the cuspid; `ex95`, `ey95`, `ez95` in meters; some relocated events may be missing).
  `mech_file`: SKHASH's mechanisms (`cuspid`, one or more rows per event; events without a mechanism are missing).
- Output: a DataFrame with one row per event of the summary file, sorted by `id`, and these columns in this order:
  `id` (the cuspid); `hyp_depth_km`, `hyp_erh_km`, `hyp_erz_km`, `hyp_rms_s` (the summary file's `DEPTH`, `ERH`,
  `ERZ` and `RMS`); `lat`, `lon`, `depth_km` (the relocation; NaN when the event is not relocated); `dz_km` (the
  relocated depth minus the HYPOINVERSE depth); `err_h95_km` (the larger of `ex95` and `ey95`, in km) and
  `err_z95_km` (`ez95` in km), NaN without a bootstrap row; `strike`, `dip`, `rake`, `quality`, `fp_uncert_deg`
  (`fault_plane_uncertainty`) and `n_pol` (`num_p_pol`) of the event's first row in the mechanism file, NaN when it
  has none; `n_solutions`, the number of rows of the event in the mechanism file (0 when none).

**Done when** `st.check("M60-T3", answer=results_report)` passes: your function on 10 made-up sets of the four
files (events not relocated, without a bootstrap row, without a mechanism and with two); then the check recomputes
the fingerprints of your copies, finds them in the run log and runs your function again on the copies.

**Stuck?** `st.hint("M60-T3", 1)`

In [ ]:
# Before you code: the Gokseong earthquakes were located with kim1983, whose top layer ends at 15 km. Which will be
# larger for these events, HYPOINVERSE's ERZ or the bootstrap's 95 % depth half-width, and by roughly what factor?
# Which of the two would you quote as "the depth uncertainty" in a paper, and what would each miss?
st.predict("M60-T3", """

""")

In [ ]:
RESULT_COLUMNS = ["id", "hyp_depth_km", "hyp_erh_km", "hyp_erz_km", "hyp_rms_s", "lat", "lon", "depth_km", "dz_km",
                  "err_h95_km", "err_z95_km", "strike", "dip", "rake", "quality", "fp_uncert_deg", "n_pol",
                  "n_solutions"]
RELOC_NAMES = ["ID", "LAT", "LON", "DEPTH", "X", "Y", "Z", "EX", "EY", "EZ", "YR", "MO", "DY", "HR", "MI", "SC", "MAG",
               "NCCP", "NCCS", "NCTP", "NCTS", "RCC", "RCT", "CID"]


def results_table(sum_file, reloc_file, boot_file, mech_file):
    """One row per located earthquake: locations, relocations, mechanisms and their uncertainties.

    sum_file, reloc_file, boot_file, mech_file : paths of the four tool files (not changed)
    returns : DataFrame with the columns RESULT_COLUMNS, one row per event of the summary file, sorted by id
    """
    # Step 1: read the summary file (strip the padded column names); keep ID-NUM, DEPTH, ERH, ERZ, RMS.
    # Step 2: read the relocations (no header) and the bootstrap errors (a comment line first); convert m to km.
    # Step 3: the first mechanism row of each cuspid, and the number of rows per cuspid.
    # Step 4: join everything to the summary rows (keep every event), add dz_km, order the columns.
    # YOUR CODE HERE
    raise NotImplementedError("M60-T3: write your code here")

Your table of the Gokseong earthquakes, saved next to the copies (nothing is printed until the check has run):

In [ ]:
table = results_table(*(PRODUCTS / name for name in sources))
table.to_csv(Path(RESULT_DIR) / "M60_results.csv", index=False)
results_report = {"results_table": results_table, "slug": PQ_SLUG, "copies": str(PRODUCTS),
                  "sha256": {key: sha256_of(PRODUCTS / name) for key, name in
                             zip(("sum", "reloc", "boot", "mech"), sources)},
                  "table": table}

In [ ]:
st.check("M60-T3", answer=results_report)

The table, rounded for reading, and the depth errors side by side:

In [ ]:
with pd.option_context("display.width", 160):
    print(table.round({"hyp_depth_km": 2, "depth_km": 3, "dz_km": 3, "err_h95_km": 4, "err_z95_km": 4,
                       "lat": 5, "lon": 5}).to_string(index=False))
ratio = table["hyp_erz_km"] / table["err_z95_km"]
print(f"\nHYPOINVERSE ERZ divided by the bootstrap's 95 % depth half-width: median {ratio.median():.0f}, "
      f"from {ratio.min():.0f} to {ratio.max():.0f}")
print(f"relocated depths: {table['depth_km'].min():.3f} to {table['depth_km'].max():.3f} km; HYPOINVERSE depths: "
      f"{table['hyp_depth_km'].min():.2f} to {table['hyp_depth_km'].max():.2f} km; kim1983's boundary: 15 km")
print("KMA catalog depths of the seven (km):", sorted(seven["depth_km"].tolist()))

**Which waves did HYPOINVERSE predict at your stations?** First its own words: the comment lines of the routine of
HYPOINVERSE that computes the travel times of a layered (CRH) model (the source code that `bin/sgtl-install tools`
compiled), then the lines of PocketQuake that choose the two models and the distance weighting, and the radius of
its station list. Then your `head_wave` of module M30 gives, for the deepest and the shallowest of your
HYPOINVERSE locations (kim1983), the **crossover distance** (glossary `crossover_distance`) per phase for the
15-km boundary and for the Moho, and counts the stations of PocketQuake's list beyond it. A station beyond a
crossover sees the head wave as its **first arrival** (glossary `first_arrival`); a NaN means that no head wave
runs along that boundary for a source at that depth.

In [ ]:
show_lines(st.pq.sgtl_home() / "tools" / "src" / "hyp1.40" / "source" / "hytrh.for",
           ["CALCULATES TRAVEL TIME", "FOR ALL ARRIVALS", "FOR REFRACTED WAVE", "DIRECT WAVE BELOW", "TDIR.GE.TMIN"])
show_lines(st.pq.pipeline_dir() / "pipeline" / "core" / "hypoinverse.py", ["CRH 1", "CRH 2", "DIS {DIS}"])
show_lines(st.pq.pipeline_dir() / "pipeline" / "config.py", ["radius_km:", "DIS: tuple"])
s_lat, s_lon = used["Latitude"].to_numpy(float), used["Longitude"].to_numpy(float)
rows = []
for label, ev in (("deepest", hypo.loc[hypo["depth"].idxmax()]), ("shallowest", hypo.loc[hypo["depth"].idxmin()])):
    away = haversine_km(float(ev["lat"]), float(ev["lon"]), s_lat, s_lon)
    for phase in ("P", "S"):
        for boundary, interface in (("15 km", 1), ("Moho", -1)):
            cross = head_wave(50.0, float(ev["depth"]), KIM1983, phase, interface)[2]
            rows.append({"event": label, "depth (km)": round(float(ev["depth"]), 2), "phase": phase,
                         "boundary": boundary, "crossover (km)": round(cross, 1),
                         "stations beyond it": int(np.sum(away > cross)),
                         "farthest station (km)": round(float(away.max()), 1)})
print(pd.DataFrame(rows).to_string(index=False))

**Explain (3-5 sentences):** Use the printed lines. How do HYPOINVERSE's ERZ and the bootstrap's 95 % depth
half-width compare, and why do they differ so much (what does each one resample or linearize)? Where do the
relocated depths lie relative to the 15-km boundary of kim1983, and what does module M30 say about the first
arrivals of a source just below a boundary? Which waves did HYPOINVERSE predict at the stations of its list for
your deepest event (read the comment lines of its routine and your crossover rows)? Which uncertainty is missing
from every number in this table?

## Part 4 · An unchanged rerun, and one changed setting (about 40 min, task 4)

HYPOINVERSE weighs every reading by a **phase weight** (glossary `phase_weight`), a code from 0 (full weight) to 4
(no weight) in column 17 (P) or 50 (S) of each phase line of the `.phs` file that PocketQuake writes for it.
PocketQuake can set the codes in two ways (`phs_weight_scheme`): by the pick probability of PhaseNet+
(`"probability"`) or by the distance of the station (`"distance"`). The settings file of your clone, and the last
lines of your cluster module (the file `st.pq.stage` wrote), say which one your cluster uses:

In [ ]:
CLUSTER_PY = st.pq.pipeline_dir() / "pipeline" / "clusters" / f"{PQ_SLUG}.py"
show_lines(st.pq.pipeline_dir() / "pipeline" / "config.py", ["phs_weight_scheme:", "phs_dist_weight_bins:",
                                                                 "(20, 0, 1)", "phs_prob_weight_bins:", "(0.90, 0)"])
print("\nlast lines of your cluster module:")
print("\n".join(CLUSTER_PY.read_text(encoding="utf-8").strip().splitlines()[-3:]))

Task 4 runs the HYPOINVERSE stage twice more: first unchanged (does it give the same bytes as your first run of
Part 2?), then with one setting changed, `phs_weight_scheme` set to `CHANGED_SCHEME`. The change goes into your
cluster module as one extra line for one run; the cell puts the file back as it was and runs the stage once more
unchanged, so that your cluster ends as it was before the change. Each run's `1.HypoInv` folder is copied right
after the run. The **deterministic computation** of module M41 (glossary `deterministic_computation`) is the
question of the first rerun.

### M60-T4 · An unchanged rerun, and the phase weights by distance · core · about 60 min

**Goal:** write `location_shift(sum_a, sum_b)`: how far each earthquake moved from one HYPOINVERSE run to another.
Then rerun the stage unchanged and with the changed weights, and report the runs in a dictionary `rerun_report`.

**Given:** the run cells below; your `haversine_km` (task M01-T1); your reader of the summary file (task 3; you may
move it into a helper function of your own and call it from both functions). Not allowed: the course package and
PocketQuake.

**Contract (the function):**
- Inputs: two paths of HYPOINVERSE summary files in PocketQuake's comma layout (task 3). Do not change them.
- Output: a DataFrame with the columns `id`, `dh_km` (the great-circle distance between the two epicenters, km, on
  the sphere of `haversine_km`), `dz_km` (depth in `sum_b` minus depth in `sum_a`, km) and `dt_s` (origin time in
  `sum_b` minus origin time in `sum_a`, seconds), one row per id found in both files, sorted by `id`. The origin
  time of a line is its date, hour and minute plus its seconds (`SEC`).

**Contract (the report):** `rerun_report = {"location_shift": location_shift, "slug": PQ_SLUG, "first": FIRST,
"rerun": RERUN, "changed": CHANGED` (the paths of the three copied folders), `"sum_sha256"` and `"phs_sha256"`:
{"first", "rerun", "changed": the SHA-256 of each copy's `kim1983/<Region>.sum` and `PHS/<Region>.phs`},
`"rerun_shift"`: your `location_shift` from the first run to the rerun, `"changed_shift"`: from the rerun to the
changed run}.

**Done when** `st.check("M60-T4", answer=rerun_report)` passes: your function on 8 made-up pairs of summary files
(moves of known size, a time across a minute, events in one file only); then the check recomputes the six
fingerprints and finds them in the run log, compares the three phase files column by column (the weight codes must
change in the changed run, and only there) and runs your function again on the copies.

**Stuck?** `st.hint("M60-T4", 1)`

In [ ]:
# Before you code: will the unchanged rerun give the same bytes as your first HYPOINVERSE run of Part 2, and why? With
# the weights by distance instead of by pick probability, which way and by how much do you expect the depths to move,
# compared with their ERZ?
st.predict("M60-T4", """

""")

In [ ]:
def location_shift(sum_a, sum_b):
    """How far each event moved between two HYPOINVERSE runs.

    sum_a, sum_b : paths of two summary files (not changed)
    returns : DataFrame with id, dh_km, dz_km, dt_s (b minus a), one row per id in both files, sorted by id
    """
    # Step 1: read both files (your reader of task 3) and build each line's origin time from date, time and SEC.
    # Step 2: keep the ids found in both; distance with your haversine_km, then depth and time differences.
    # YOUR CODE HERE
    raise NotImplementedError("M60-T4: write your code here")

**The first run's folder** (copied now, before anything else runs) and **the unchanged rerun** (a few seconds):

In [ ]:
def copy_hypoinverse(name):
    """Copy the cluster's 1.HypoInv folder to results/<name> (replacing an older copy); return the copy's path."""
    target = Path(RESULT_DIR) / name
    if target.exists():
        shutil.rmtree(target)
    shutil.copytree(RUN_DIR / "1.HypoInv", target)
    return target


FIRST = copy_hypoinverse("M60_hypoinverse_first")
assert st.pq.run(PQ_SLUG, "hypoinverse", "hypoinverse", cores=1, mem_gb=PQ_MEM_GB, apply=True) == 0
RERUN = copy_hypoinverse("M60_hypoinverse_rerun")
print("copied to", FIRST, "and", RERUN)

**The one change**, then the module put back and one more unchanged run (its output is kept for later: the last
line names the full log):

In [ ]:
module_text = CLUSTER_PY.read_text(encoding="utf-8")
CHANGE = f'CONFIG = replace(CONFIG, phs_weight_scheme="{CHANGED_SCHEME}")   # M60: one run only\n'
print("the line this cell adds for one run:", CHANGE.strip())
quiet = io.StringIO()
try:
    CLUSTER_PY.write_text(module_text + "\n" + CHANGE, encoding="utf-8")
    with contextlib.redirect_stdout(quiet):
        changed_rc = st.pq.run(PQ_SLUG, "hypoinverse", "hypoinverse", cores=1, mem_gb=PQ_MEM_GB, apply=True)
finally:
    CLUSTER_PY.write_text(module_text, encoding="utf-8")
print(quiet.getvalue().strip().splitlines()[-1])
assert changed_rc == 0, quiet.getvalue()[-2000:]
CHANGED = copy_hypoinverse("M60_hypoinverse_changed")
print("copied to", CHANGED)
print("cluster module restored:", sha256_of(CLUSTER_PY) == hashlib.sha256(module_text.encode("utf-8")).hexdigest())
with contextlib.redirect_stdout(io.StringIO()):
    assert st.pq.run(PQ_SLUG, "hypoinverse", "hypoinverse", cores=1, mem_gb=PQ_MEM_GB, apply=True) == 0
print("one more unchanged run: done")

The report (nothing is printed until the check has run):

In [ ]:
SUM_NAME, PHS_NAME = f"kim1983/{REGION}.sum", f"PHS/{REGION}.phs"
rerun_report = {"location_shift": location_shift, "slug": PQ_SLUG, "first": str(FIRST), "rerun": str(RERUN),
                "changed": str(CHANGED),
                "sum_sha256": {k: sha256_of(Path(f) / SUM_NAME) for k, f in
                               (("first", FIRST), ("rerun", RERUN), ("changed", CHANGED))},
                "phs_sha256": {k: sha256_of(Path(f) / PHS_NAME) for k, f in
                               (("first", FIRST), ("rerun", RERUN), ("changed", CHANGED))},
                "rerun_shift": location_shift(FIRST / SUM_NAME, RERUN / SUM_NAME),
                "changed_shift": location_shift(RERUN / SUM_NAME, CHANGED / SUM_NAME)}

In [ ]:
st.check("M60-T4", answer=rerun_report)

Now measure. First the rerun: the same bytes? If not, which columns of which phase lines differ, and by how much.
Then what the stages in between wrote, by their own words (the first lines of two functions of PocketQuake), and
how many of the run's SAC files changed after your first HYPOINVERSE run.

In [ ]:
print("rerun's summary file the same bytes as the first run's:",
      rerun_report["sum_sha256"]["rerun"] == rerun_report["sum_sha256"]["first"])
lines = {name: (Path(folder) / PHS_NAME).read_text(encoding="utf-8").splitlines()
         for name, folder in (("first", FIRST), ("rerun", RERUN), ("changed", CHANGED))}
for a, b in (("first", "rerun"), ("rerun", "changed")):
    cols, kinds, example = {}, {}, None
    for u, v in zip(lines[a], lines[b]):
        diff = [i + 1 for i, (p, q) in enumerate(zip(u, v)) if p != q]
        for i in diff:
            cols[i] = cols.get(i, 0) + 1
        if diff:
            kind = "S line" if u[47:49] == "ES" or u[46:48] == "ES" else ("P line" if u[13:15] == "IP" else "other")
            kinds[kind] = kinds.get(kind, 0) + 1
            example = example or (u, v)
    print(f"\n{a} -> {b}: {sum(kinds.values())} of {len(lines[a])} phase lines differ; by kind {kinds}; "
          f"columns (1-based) and how many lines: {dict(sorted(cols.items()))}")
    if example:
        print("  for example\n  " + example[0] + "\n  " + example[1])
print()
show_lines(st.pq.pipeline_dir() / "pipeline" / "core" / "hypoinverse.py", ["picks stored in the SAC a/t0 headers"])
show_lines(st.pq.pipeline_dir() / "pipeline" / "core" / "rereference.py", ["Rewrite every run-tree event's SAC"])
first_time = (FIRST / SUM_NAME).stat().st_mtime
sacs = list((RUN_DIR / "waveforms_100km").rglob("*.sac"))
print(f"SAC files of the run changed after your first HYPOINVERSE run: "
      f"{sum(p.stat().st_mtime > first_time for p in sacs)} of {len(sacs)}")
for line in RUNS_LOG.read_text(encoding="utf-8").splitlines():
    run = json.loads(line)
    if run.get("cluster") == PQ_SLUG:
        print(run["ts"], run["command"], run.get("stage_from", run.get("branch", "")), "->",
              run.get("through", ""), f"{run['seconds']:.0f} s")

And what each run moved, next to the errors of the first run:

In [ ]:
first_sum = pd.read_csv(FIRST / SUM_NAME, skipinitialspace=True).rename(columns=lambda c: c.strip())
moves = rerun_report["rerun_shift"].merge(rerun_report["changed_shift"], on="id", suffixes=("_rerun", "_changed"))
moves = moves.merge(first_sum[["ID-NUM", "ERH", "ERZ"]].rename(columns={"ID-NUM": "id"}), on="id")
print(moves.round(3).to_string(index=False))
print("\nweights by distance: depth change from", round(moves["dz_km_changed"].min(), 2), "to",
      round(moves["dz_km_changed"].max(), 2), "km; median |dz| / ERZ:",
      round(float((moves["dz_km_changed"].abs() / moves["ERZ"]).median()), 2))

**Explain (3-5 sentences):** Use the printed lines. Was the unchanged rerun the same bytes? If not, which phase
lines and columns differed, by how much, and which stage that ran in between explains it (quote the two lines and
the count of changed SAC files; if they do not settle it, say what would)? What did the weights by distance move
(the horizontal, depth and time columns), and are the changes larger or smaller than the first run's ERH and ERZ?
What does that say about how much a depth of this table depends on one choice of weighting?

## Part 5 · The data of your project: an inventory (about 60 min, task 5)

This part inventories the Suncheon research track: the named catalog subset `suncheon` of the course
(`data/manifests/sequences.yaml`: every KMA event within a radius of a center, between two dates) and its STP
dataset `suncheon_area`. That dataset is large (the dry run below says how large), so this module does not fetch it:
the waveforms belong to the project weeks. Here you plan. A **data inventory** (glossary `data_inventory`) says,
before any download, which events there are, which stations were running when each one happened (the **station
coverage**, glossary `station_coverage`), and what the download will cost.

Two station lists are at hand, and they are not the same:

- **The national list of STP**: every dataset folder that `sgtl data fetch` made holds STP's station list in
  `station_table/KS_station.csv` and `station_table/KG_station.csv` (code, coordinates, elevation). It is the list
  of today, with no dates, and the same in every dataset folder; your Gokseong folder has one.
- **The dated list of the course**: `st.data.stations()` (`data/stations/stations.csv`, its README in the same
  folder) gives each station a `start_date`, the first day of the station in its source list. A station's life
  can have several **station epochs** (glossary `station_epoch`; module M03), and this list has no end dates. The
  full epochs are in the KS/KG station metadata compiled by Byeong-Seok Ahn (Pukyong National University; Ahn et al.
  2024), which `bin/sgtl-install metadata` installed: the project weeks use it, with that acknowledgment.

First your project folder: the next cell copies the project templates into `project/` once (the same as
`sgtl new project` in a terminal; a file you already have is never touched).

Then the inputs of task 5: the subset's settings, the catalog subset and the dated station list.

In [ ]:
from sgtl_tutorial.cli import new_project

new_project()
print()
seq_spec = st.data.sequences()[PROJECT_SEQUENCE]
print({k: seq_spec[k] for k in ("description", "center_lat", "center_lon", "radius_km", "start_utc", "end_utc")})
events = st.data.sequence(PROJECT_SEQUENCE)
dated = st.data.stations()
print(f"{len(events)} catalog events in the subset {PROJECT_SEQUENCE!r}; {len(dated)} stations in the dated list")
print(events[["event_id", "origin_utc", "latitude", "longitude", "depth_km", "ml"]].head(3).to_string(index=False))
print(dated.head(3).to_string(index=False))

And the download plan of the project dataset (a dry run: nothing is downloaded, and the STOP lines about your STP
account do not matter here):

In [ ]:
st.datafetch.fetch(PROJECT_DATASET)

### M60-T5 · The inventory of the Suncheon earthquakes · core · about 60 min

**Goal:** write `inventory(events, stations, radii_km=(30.0, 50.0, 100.0), years=None)`: for every event, how
many stations of a dated list were running within each radius of its epicenter; and per year, the events and the
worst station coverage.

**Given:** Part 5; your `haversine_km` (task M01-T1); pandas. Not allowed: the course package, PocketQuake and
geodesy libraries (pyproj, obspy.geodetics, geographiclib, geopy and the like): distances come from your
`haversine_km`.

**Contract:**
- Inputs: `events`, a DataFrame like `st.data.sequence(...)` (columns `event_id`, `origin_utc` in UTC, `latitude`,
  `longitude`, `depth_km` (NaN when the catalog has no depth) and `ml`); `stations`, a DataFrame like
  `st.data.stations()` (columns `network`, `station`, `latitude`, `longitude`, `start_date`, NaT when unknown);
  `radii_km`, the radii in km; `years`, the years of the yearly table, or None for every year from the year of the
  first event to the year of the last one. Do not change the inputs.
- A station counts for an event within R km when its `start_date` (midnight UTC of that day) is on or before the
  event's origin time and its distance from the event's epicenter (your `haversine_km`) is at most R. A station
  with no start date never counts.
- Output: a tuple `(per_event, per_year)`.
  `per_event`: one row per event, sorted by `origin_utc` (then `event_id`), a fresh index, with the columns
  `event_id`, `origin_utc`, `year` (the UTC year), `latitude`, `longitude`, `depth_km`, `ml`, then one column per
  radius in the order given, named `f"n_{R:g}km"` (for example `n_30km`), with the station counts (whole numbers).
  `per_year`: one row per year asked for, sorted, the index named `year`, with the columns `n_events` (events of
  that year), `n_depth` (those with a catalog depth), `ml_max` (the largest `ml`; NaN without events), then one
  column per radius named `f"n_{R:g}km_min"`: the fewest stations that any event of that year had (NaN without
  events).

**Done when** `st.check("M60-T5", answer=inventory_report)` passes: your function on made-up events and stations
(stations near each radius, started on the day of an event, the day after, or never dated; years without events);
then the check runs your function again on the subset and the dated list, reads your two saved files, recomputes
the download plan and counts the national list itself.

**Stuck?** `st.hint("M60-T5", 1)`

In [ ]:
# Before you code: the subset spans the years printed above its settings (2002-2026). For the worst-covered event of
# 2012 and of 2019, how many stations within 100 km do you expect, and within 30 km? What would
# a difference of that size do to the depth of an event located with the stations of its own time?
st.predict("M60-T5", """

""")

In [ ]:
def inventory(events, stations, radii_km=(30.0, 50.0, 100.0), years=None):
    """Station coverage of every event and of every year, from a dated station list.

    events   : DataFrame like st.data.sequence(...) (not changed)
    stations : DataFrame like st.data.stations() (not changed)
    radii_km : radii in km; years : years of the yearly table, or None
    returns  : (per_event, per_year) as in the Contract
    """
    # Step 1: the events sorted by time; their origin times and the stations' start dates on one clock (UTC).
    # Step 2: per event and radius, the stations running at the origin time and within the radius (haversine_km).
    # Step 3: per year, the counts, the largest magnitude and the fewest stations of any event.
    # YOUR CODE HERE
    raise NotImplementedError("M60-T5: write your code here")

Your inventory of the subset, saved into your project folder; then the plan's numbers and the national list within
the plan's radius of the dataset's center (nothing is printed until the check has run):

In [ ]:
per_event, per_year = inventory(events, dated, radii_km=RADII_KM, years=YEARS)
INVENTORY = PROJECT / "inventory"
INVENTORY.mkdir(parents=True, exist_ok=True)
per_event.to_csv(INVENTORY / "events.csv", index=False)
per_year.to_csv(INVENTORY / "years.csv")
ds = st.datafetch.dataset(PROJECT_DATASET)
plan = st.datafetch.make_plan(ds)
national = pd.concat([pd.read_csv(FOLDER / "station_table" / f"{net}_station.csv", dtype={"Code": str})
                      for net in ("KS", "KG")], ignore_index=True)
near = national[haversine_km(ds.center[0], ds.center[1], national["Latitude"].to_numpy(float),
                             national["Longitude"].to_numpy(float)) <= ds.radius_km]
in_dated = set(zip(dated["network"], dated["station"]))
undated = [(n, c) for n, c in zip(near["Network"], near["Code"]) if (n, c) not in in_dated]
inventory_report = {"inventory": inventory, "sequence": PROJECT_SEQUENCE, "radii_km": RADII_KM, "years": YEARS,
                    "per_event": per_event, "per_year": per_year,
                    "files": {"events": str(INVENTORY / "events.csv"), "years": str(INVENTORY / "years.csv")},
                    "plan": {"dataset": PROJECT_DATASET, "events": len(ds.events), "pending": len(plan.pending),
                             "stations": plan.n_stations, "requests": plan.total_requests,
                             "sessions": len(plan.sessions)},
                    "national": {"folder": str(FOLDER / "station_table"), "within": len(near),
                                 "undated": len(undated)}}

In [ ]:
st.check("M60-T5", answer=inventory_report)

The yearly table, and the station lists side by side:

In [ ]:
with pd.option_context("display.width", 160):
    print(per_year.to_string())
print(f"\nthe plan of {PROJECT_DATASET}: {inventory_report['plan']}")
print(f"national list (STP, today): {len(near)} stations within {ds.radius_km:g} km of the dataset's center "
      f"{tuple(round(c, 3) for c in ds.center)}; {len(undated)} of them are not in the dated list")
print("events without a catalog depth:", int(per_event["depth_km"].isna().sum()), "of", len(per_event))
depths = per_event["depth_km"].dropna()
print("catalog depths of the subset, km: number of events", depths.value_counts().sort_index().to_dict())
deep_rows = per_event[per_event["depth_km"] >= 20.0]
print(f"events at 20 km or deeper: {len(deep_rows)} of {len(depths)} with a depth, on "
      f"{deep_rows['origin_utc'].dt.date.nunique()} different UTC days")

A map of the subset and of the dated stations within 100 km of its center, colored by the year they started:

In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 5.5))
close = dated[haversine_km(seq_spec["center_lat"], seq_spec["center_lon"], dated["latitude"].to_numpy(float),
                           dated["longitude"].to_numpy(float)) <= 100.0]
mark = ax.scatter(close["longitude"], close["latitude"], c=close["start_date"].dt.year, cmap="viridis", marker="^",
                  s=40, edgecolor="k", linewidth=0.4, label="station (dated list)")
ax.scatter(events["longitude"], events["latitude"], s=10, color="crimson", label="Suncheon event (KMA)")
fig.colorbar(mark, ax=ax, label="start year of the station")
ax.set_xlabel("Longitude (°E)")
ax.set_ylabel("Latitude (°N)")
ax.set_aspect(1.0 / np.cos(np.radians(seq_spec["center_lat"])))
ax.legend(loc="lower left", bbox_to_anchor=(0.0, 1.0), ncol=2, frameon=False)
fig.savefig(Path(FIGURE_DIR) / "M60_inventory_map.png", dpi=150, bbox_inches="tight")
plt.show()

**Explain (3-5 sentences):** Use the printed table. How many stations within 30 km and within 100 km did the
worst-covered event have in 2012, and in 2019 (read the rows)? Which years have events but no station within 30 km,
and what does that mean for comparing the depths of early and recent events (hypothesis H1 of your plan)? How do the
plan's station number, the national list and the dated list compare, and why can the counts of early years be wrong
in either direction (read the README of the dated list)? Where will you get better station dates in the project?

## Part 6 · Testable claims and a frozen plan (about 120 min, task 6)

One paper is about these earthquakes: Kwak et al. (2022) (the reading of this module). A project starts from what is
known, written so that it can be checked. A **claims table** (glossary `claims_table`) lists every claim of the
paper as a **testable claim** (glossary `testable_claim`): one sentence that a measurement could prove wrong, with
the quantity and value the paper gives (when it gives one), the test that would check it with our data, the data
that test needs, and the result that would refute it. A claim such as "the events are deep" is not testable; "the
relocated events lie at about X km depth" is, once you say which result would refute it.

Your plan starts from two questions, each raised by a source that the template names. **H1** comes from the KMA
catalog: its depths for the subset (the counts printed in Part 5) suggest a main group near 10 km and a smaller set
at 20 km or deeper. Are there two depth populations, and are both real? **H2** comes from the same catalog and from
Kwak et al. (2022): is the persistent cluster about 10 km west-northwest of Suncheon one plane, and if so, what are
its strike and dip? You turn each into a **hypothesis** (glossary `hypothesis`: a statement about nature that a test
can support or reject) in your own words. Before you look at any result, you write an **analysis plan** (glossary
`analysis_plan`): the questions, the tests, the **pass criteria** (glossary `pass_criteria`: the numbers that decide
each test), what would change the conclusion, and your **prior exposure** (glossary `prior_exposure`: what you have
already seen that could shape your choices). Writing the plan first and freezing it is the idea of a
preregistration (Nosek et al. 2018): it keeps you from choosing a test after you have seen which one gives the
answer you hoped for. A later change is allowed as a **plan amendment** (glossary `plan_amendment`): dated, with the
reason, and whether you had seen results.

The freeze is recorded by an **MD5** checksum (glossary `md5`) of `criteria.yaml`: 32 hexadecimal characters
computed from the file's bytes (an older and shorter fingerprint than SHA-256; enough to show that a file has not
changed, not a protection against forgery). It goes into the plan and into your **lab book** (glossary `lab_book`,
`LABBOOK.md`: a dated record of what you ran and what came out). Before the freeze, your mentor reads the plan and
writes the **sign-off** (glossary `sign_off`): the plan's line `- Mentor sign-off: <name>, <YYYY-MM-DD>`, dated on
or before the freeze. A plan without it is not frozen.

The steps, in your project folder (`project/`, copied in Part 5; read its README first):

1. `claims.csv`: one row per claim of Kwak et al. (2022), at least six; `source` is the reference key `kwak2022`
   (a key of `references/references.yaml`), `where` the place in the paper (a section, figure, table or page
   number, or `abstract`). A claim you could read only in the abstract gets `abstract`, and a line in `LABBOOK.md`
   that it must be checked against the full paper before the project tests it.
2. `ANALYSIS_PLAN.md`: answer every question (the lines that start with `>`; keep them as they are);
   `criteria.yaml`: fill every null (a threshold is a number and its unit).
3. The sign-off: your mentor reads the plan and criteria and writes the sign-off line.
4. The freeze: set `frozen` in `criteria.yaml`, run the next cell, and write the checksum it prints into the plan's
   line "MD5 checksum of `criteria.yaml` at the freeze" and into `LABBOOK.md`; set "Frozen on" to the same date.
   Any later change of `criteria.yaml` changes its checksum: then add an amendment.

The reference keys of the course, and the checksum of your `criteria.yaml` as it is now:

In [ ]:
REFERENCE_KEYS = sorted(entry["key"] for entry in
                        yaml.safe_load(st.data.repo_path("references", "references.yaml").read_text(encoding="utf-8")))
print(len(REFERENCE_KEYS), "reference keys, for example", REFERENCE_KEYS[:3], "and kwak2022:", "kwak2022" in REFERENCE_KEYS)
print("MD5 of project/criteria.yaml now:", hashlib.md5((PROJECT / "criteria.yaml").read_bytes()).hexdigest())

### M60-T6 · An audit of your project folder · core · about 120 min

**Goal:** write `project_audit(project_dir, reference_keys, min_claims=6, min_numbers=3, min_words=4,
min_answer_words=12)`: six checks of a project folder, so that a plan cannot be frozen with a hole in it or with
text that only fills a hole. Then write your claims table and your plan, get the sign-off, freeze them, and run the
audit on your folder.

**Given:** Part 6; the templates in `project/`; `yaml.safe_load`, `hashlib.md5`, pandas. Not allowed: the course
package.

**Contract:**
- Inputs: `project_dir`, a folder (a `Path` or a string) with `claims.csv`, `criteria.yaml`, `ANALYSIS_PLAN.md`
  and `LABBOOK.md` as in the templates (any of them may be missing). Do not change it. `reference_keys`, the allowed
  `source` keys; `min_claims`, `min_numbers`, `min_words`, `min_answer_words`: whole numbers.
- A **word** here is a piece of a text between spaces (as `str.split()` cuts it) with at least two letters in it
  (letters as `str.isalpha` sees them, one character at a time): `km`, `NE-SW` and `first-motion` are words; `x`,
  `3`, `-` and `P` are not. A **real date** is a text `YYYY-MM-DD` that is a day of the calendar.
- Output: a DataFrame with the columns `item`, `status` (`"PASS"` or `"FAIL"`) and `detail` (a short text of your
  choice), one row per check in this order:
  1. `claims`: `claims.csv` reads as a table (every cell as text; an empty cell is empty text) with exactly the
     template's eleven columns in its order; it has at least `min_claims` rows; `claim_id` is unique; in every row
     `claim_id`, `source`, `where`, `topic`, `statement`, `test`, `data_needed` and `refuted_if` are not empty
     (spaces alone count as empty); every `source` is one of `reference_keys`; in every row `statement`, `test`,
     `data_needed` and `refuted_if` have at least `min_words` words each; and every `where` holds a digit (a
     section, figure, table or page number) or the word `abstract` in any case.
  2. `numbers`: every row whose `reported_value` is not empty has a finite number there (as `float` reads it), and
     a non-empty `quantity` and `unit`; at least `min_numbers` rows have a `reported_value`. FAIL when `claims.csv`
     cannot be read with the template's columns.
  3. `criteria`: `criteria.yaml` reads as YAML into a mapping with the keys `project`, `author`, `frozen` and
     `hypotheses`; `hypotheses` maps `H1` and `H2` each to a mapping with the keys `statement`, `test`, `events`,
     `metric`, `threshold` and `decision`; no value anywhere in the file is null or empty text; `frozen`, written as
     text (`str`), is a real date; in `H1` and in `H2`, `statement`, `test`, `events`, `metric` and `decision`,
     written as text, have at least `min_words` words each; and each `threshold`, written as text, is a number and
     its unit: its first piece (`split(None, 1)`) is a finite number as `float` reads it, and the rest holds at
     least one word.
  4. `freeze`: the MD5 (as `hexdigest()` writes it) of the bytes of `criteria.yaml` is on the plan's line that
     starts with ``- MD5 checksum of `criteria.yaml` at the freeze:`` and anywhere in `LABBOOK.md`; the text after
     `- Frozen on:` on the plan's line (spaces stripped) equals `frozen` written as text; and the plan has a line
     that starts with `- Mentor sign-off:` whose text after the colon, cut at its last comma, gives a name with at
     least one word and (spaces stripped) a real date that is not later than `frozen`. FAIL when a file is missing
     or `criteria.yaml` has no `frozen`.
  5. `plan`: `ANALYSIS_PLAN.md` and `LABBOOK.md` exist and have no `[write here` left anywhere, and every question
     of the plan has an answer of at least `min_answer_words` words. A question is a run of lines, one after the
     other, that start with `>`; its answer is every line after it up to the next line that starts with `>` or `#`
     (or the end of the file).
  6. `amendments`: the plan has a heading line `## Amendments`; every row of the table under it (the lines that
     start with `|`, after its header line and its separator line) has four cells, none empty, and the first is a
     real date on or after the date `frozen` (FAIL for any row when `frozen` is not a real date). No rows: PASS.

**Done when** `st.check("M60-T6", answer=project_report)` passes: your function on 29 made-up project folders, each
built with one known problem or none; then the check recomputes the fingerprints of your four files, runs your
function again on your folder (every row must PASS), counts your claims of `kwak2022`, and reads the rules 1, 3, 4
and 5 in your files itself, together with the questions of the template (your plan keeps all of them, in order).

**Stuck?** `st.hint("M60-T6", 1)`

In [ ]:
# Before you code: Kwak et al. (2022) give, in their abstract, how many KMA catalog earthquakes their study period
# holds. The course's subset `suncheon` is a circle of 10 km around a point near the city (Part 5 printed it). Do you
# expect the subset to hold more events of the paper's period than the paper counts, as many, or fewer? What share of
# them do you expect KMA to have entered into its catalog in 2022 or later, the year the paper appeared? Give two
# reasons why the two counts could differ.
st.predict("M60-T6", """

""")

In [ ]:
CLAIM_COLUMNS = ["claim_id", "source", "where", "topic", "statement", "quantity", "reported_value", "unit", "test",
                 "data_needed", "refuted_if"]


def project_audit(project_dir, reference_keys, min_claims=6, min_numbers=3, min_words=4, min_answer_words=12):
    """Six checks of a project folder before (and after) its plan is frozen.

    project_dir    : folder with claims.csv, criteria.yaml, ANALYSIS_PLAN.md and LABBOOK.md (not changed)
    reference_keys : the allowed source keys
    min_claims, min_numbers, min_words, min_answer_words : whole numbers, as in the Contract
    returns        : DataFrame with item, status, detail; six rows in the Contract's order
    """
    # Step 1: the claims table as text; its columns, rows, empty cells, sources, words, places and numbers.
    # Step 2: criteria.yaml as YAML; walk every value (mappings inside mappings) and look for null or empty text;
    #         then the words of the entries of H1 and H2 and the number and unit of each threshold.
    # Step 3: the MD5 of criteria.yaml's bytes, the header lines of the plan (with the sign-off) and the lab book.
    # Step 4: the placeholders, the questions and their answers, and the rows of the amendments table.
    # YOUR CODE HERE
    raise NotImplementedError("M60-T6: write your code here")

Your audit of your project folder (nothing is printed until the check has run):

In [ ]:
audit = project_audit(PROJECT, REFERENCE_KEYS)
files = ("claims.csv", "criteria.yaml", "ANALYSIS_PLAN.md", "LABBOOK.md")
project_report = {"project_audit": project_audit, "project": str(PROJECT), "reference_keys": REFERENCE_KEYS,
                  "criteria_md5": hashlib.md5((PROJECT / "criteria.yaml").read_bytes()).hexdigest(),
                  "sha256": {name: sha256_of(PROJECT / name) for name in files}, "audit": audit}

In [ ]:
st.check("M60-T6", answer=project_report)

Your audit, then the catalog against the first claim. A KMA event ID starts with four digits that look like a year.
The next cell first compares them, over the whole catalog that ships with the course, with the year of the origin
time on the Korean clock (KST, the clock of KMA): how many IDs carry the same year, how many an earlier or a later
one, and by how many years. Where an ID carries a year several years after the origin, a likely reading is that KMA
entered the event in that later year; the catalog does not say so, so treat it as a pattern with its printed
exceptions. Then the subset's events in the paper's period (September 2009 to August 2020, UTC): how many carry an
ID year of 2022 or later, which years those are, and where the events lie, in cells of 0.05°:

In [ ]:
print(audit.to_string(index=False))
id_year = kma["event_id"].astype(str).str[:4].astype(int)
kst_year = kma["origin_utc"].dt.tz_convert("Asia/Seoul").dt.year
lag = id_year - kst_year
print(f"\nthe whole catalog: {len(kma)} events; ID year = KST year of the origin: {int((lag == 0).sum())}; "
      f"earlier: {int((lag < 0).sum())}; later: {int((lag > 0).sum())}")
print("ID year minus KST year, where they differ (years: events):", lag[lag != 0].value_counts().sort_index().to_dict())
for label, mask in (("earlier", lag < 0), ("later by 1 year", lag == 1), ("later by 5 or more years", lag >= 5)):
    print(f"  KST years of the origins whose ID year is {label}:", kst_year[mask].value_counts().sort_index().to_dict())
period = events[(events["origin_utc"] >= pd.Timestamp("2009-09-01", tz="UTC"))
                & (events["origin_utc"] < pd.Timestamp("2020-09-01", tz="UTC"))]
entry = period["event_id"].str[:4].astype(int)
same = entry == period["origin_utc"].dt.tz_convert("Asia/Seoul").dt.year
recent = entry >= 2022
print(f"\n{len(period)} subset events in the paper's period: ID year = KST year for {int(same.sum())}; "
      f"an ID year of 2022 or later for {int(recent.sum())} ({recent.mean():.0%}); those ID years: "
      f"{entry[recent].value_counts().sort_index().to_dict()}")
cells = period.assign(lat=(period["latitude"] / 0.05).round() * 0.05, lon=(period["longitude"] / 0.05).round() * 0.05,
                      recent=recent)
print(cells.groupby(["lat", "lon"]).agg(events=("event_id", "size"), id_2022_on=("recent", "sum"),
                                        depth_median_km=("depth_km", "median"),
                                        depth_known=("depth_km", "count")).round(2).to_string())

**Explain (3-5 sentences):** Use the printed lines. How closely does the ID year follow the KST year over the whole
catalog (read the counts and the exceptions), and what does that let you say, and not say, about when KMA entered an
event? How many subset events fall in the paper's period, how many of them carry an ID year of 2022 or later, and how
does that compare with your prediction and with the number in the paper's abstract? Which cell holds the cluster the
paper studies, and what do the other cells add to your project (read their depths)? Which of your claims can the
project test with the stations of each era (task 5), and which not? Which entry of your `criteria.yaml` was hardest
to fix before seeing results, and why?

## Next · Two required research tracks

Keep the signed Suncheon plan and its M60 audit record. Complete its `TEMPLATE_MATCHING_PLAN.md` extension before
evaluating detection outcomes, using a dated amendment if the original plan was already frozen. Then complete
the separate `project/sangju/ANALYSIS_PLAN.md`, `DATA_PLAN.md` and `criteria.yaml`, with independent inputs and
mentor sign-off before research results are evaluated. `sgtl new project` adds any missing child templates and
keeps existing Suncheon work.

Use the required M22 methods in both tracks, including independent validation and cross-template duplicate
merging. The [project overview](../../../docs/project.md) and [Sangju guide](../../../docs/sangju_project.md) describe the
data requests, reference-region comparisons, uncertainty tests and two reports. The M60-T6 PASS covers its
Suncheon audit contract; completion of the portfolio also requires the separate Sangju evidence review.